# ema-first-moment — worked example 3: First-moment EMA with bias correction

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-first-moment`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Because the EMA buffer starts at zero, early estimates are biased toward zero. Adam corrects this by dividing by `1 - beta1**t` at step `t`. The raw EMA update is unchanged; the correction is applied to a *copy* used for the parameter step, leaving the buffer itself un-corrected.

## Worked solution

We run the EMA and produce the bias-corrected estimate at each step.

1. Keep a step counter `step`, starting at 1 on the first update (the exponent in the correction is the 1-based step index).
2. Update the buffer in place: `m.copy_(beta1*m + (1-beta1)*g)`.
3. Compute the correction divisor `1 - beta1**step`. The corrected estimate is `m / (1 - beta1**step)`. We do NOT write this back into `m` — the buffer must stay the raw EMA so future steps recurse correctly.
4. Early on, the divisor is small (e.g. `1 - 0.9 = 0.1`), so it scales the under-estimated `m` up sharply; as `step` grows the divisor approaches 1 and the correction fades. We print both raw and corrected values to show the gap shrinking.

In [ ]:
import torch as t

t.manual_seed(2)
beta1 = 0.9
m = t.zeros(2)
grads = [t.tensor([1.0, 1.0])] * 4

def ema_m_corrected(m, g, beta1, step):
    m.copy_(beta1 * m + (1 - beta1) * g)
    m_hat = m / (1 - beta1 ** step)
    return m_hat

for step, g in enumerate(grads, start=1):
    m_hat = ema_m_corrected(m, g, beta1, step)
    print(f'step {step}: raw={m.tolist()} corrected={[round(v,4) for v in m_hat.tolist()]}')
print('corrected approaches 1.0 for constant grad:', bool(m_hat[0] > m[0]))